# <u>Feature Selection</u>

### Prerequisites:

* <a href="../1.Supervised Learning/1.Regression/">Check out the notebookes on Regression</a>
* <a href="../1.Supervised Learning/2.Classification/">Check out the notebooks on Classification</a>

## Topics

* [1. Goal](#goal)
* [2. Motivation](#motivate)
    * [2.1 Size of Datasets](#size)
* [3. Feature selection vs. Feature extraction](#selection_extraction)
* [4. Types of feature selection](#types)
    * [4.1 Filter Methods](#filter)
    * [4.2 Wrapper Methods](#wrapper)
    * [4.3 Embedded Methods](#embedded)
* [5. Key Insights](#insights)
* [6. Feature Selection & Statistical Inference](#selection_inference)
* [7. How Filter Methods Work (Pipeline)](#pipeline)
* [8. Overall Takeaways](#overall)


In [45]:
import numpy as np # for arrays and random numbers

import random # provides functions for generating pseudo-random numbers and making random selections

import pandas as pd # for dataframes

import matplotlib.pyplot as plt # for plotting

from sklearn.feature_selection import chi2 # chi square test (categorical features)

from sklearn.preprocessing import KBinsDiscretizer # converts continuous numerical features into discrete bins (intervals)

from scipy.stats import ttest_ind # welch t test

from sklearn.feature_selection import f_classif # F test

from sklearn.feature_selection import mutual_info_classif # mutual information
from sklearn.metrics import mutual_info_score # mutual information between discrete labels

from sklearn.metrics import roc_auc_score # AUC / ROC (per feature)

from sklearn.feature_selection import SelectKBest # Selecting top features

from sklearn.feature_selection import SequentialFeatureSelector as SFS # Wrapper Methods (Forward / Backward Selection)

from sklearn.feature_selection import RFE # Recursive Feature Elimination (RFE)

#from mlxtend.feature_selection import SequentialFeatureSelector as SFS

#from deap import base, creator, tools # Genetic Algorithms

from sklearn.pipeline import Pipeline # pipeline

from sklearn.model_selection import cross_val_score # k-gold CV

from sklearn.model_selection import GridSearchCV # Grid search

# ML models
from sklearn.linear_model import Lasso # Lasso regression
from sklearn.neighbors import KNeighborsClassifier # k neares neighbors
from sklearn.preprocessing import PolynomialFeatures # preprocessing for Polynomial regression
from sklearn.linear_model import LinearRegression # perform OLS
from sklearn.tree import DecisionTreeClassifier # for Classification trees
from sklearn.tree import DecisionTreeRegressor # for Regression trees
from sklearn.linear_model import Ridge # Ridge regression
from sklearn.svm import SVC # (non) linear (non)separable SVM
from sklearn.linear_model import LogisticRegression # logistic regression
from sklearn.naive_bayes import GaussianNB # Naive Bayes
from sklearn.ensemble import RandomForestClassifier # Random forest classifier

from sklearn.datasets import load_breast_cancer # dataset
from sklearn.datasets import make_classification # toy data for classification

print("Setup complete")

Setup complete


<a class="anchor" id="goal"></a>
# 1. Goal

**Find a well-performing, hopefully small set of features.**

Feature selection is critical for:
- reducign noise and overfitting
- improving performance/generalization
- enhancing interpretability by identifying most imformative features

#### Methods: Select features via <u>domain knowledge</u> or <u>algorithmic approaches</u>

<a class="anchor" id="motivate"></a>
# 2. Motivation

* Model is not harmed by irrelevant features since their parameters can simply be estimated as 0
* However in practice, irrelevant features can worsen generalization
- Having redundant features can cost money or time during prediction
- Many models require that sample size $n$ be greater than the number of features $(n > p)$


<a class="anchor" id="size"></a>
## 2.1 Size of Datasets

- Classical setting: If we have around $100$ features $\Rightarrow$ Feature selection might be relevant

- Medium: If we have between $100$ and $1000$ features  $\Rightarrow$ Feature selection often helpful

- High-dimensions: With more than $1000$ features $\Rightarrow$ Feature selection cruicial

<a class="anchor" id="selection_extraction"></a>
# 3. Feature selection vs. Feature extraction


<div style="display:flex; gap:20px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">


<h5 style="text-align:center;">Feature selection</h5>

- select a subset $\tilde{p}$ of $p$ features $(\tilde{p} < p)$
- keeps interpretability

<p align="center">
<img src="pics/50.png" width="600"/>
</p>


</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">

<h5 style="text-align:center;">Feature extraction</h5>

- transform $p$ features to $\tilde{p}$ with (linear / nonlinear combinations)
- information on individual features can be lost
- Example: Unsupervised PCA (multidimensional scaling,manifold learning), Supervised PCA (Partial least squares)

<p align="center">
<img src="pics/51.png" width="600"/>
</p>

</div>
</div>



<a class="anchor" id="types"></a>
# 4. Types of feature selection

<a class="anchor" id="filter"></a>
## 4.1 Filter Methods

**Evaluate relevance of features via statistics such as correlation with target variable.**

#### Mechanism of Filter Methods:
Compute relevance score for each feature $x_j$ (measure that quantifies the dependancy between features and the target variable), rank them then select top $\tilde{p}$ features. Train model on top features.


#### How to choose $\tilde{p}$:
- Arbitrarily select $\tilde{p}$ 
- Treat $\tilde{p}$  as hyperparameter and tune in a pipeline based on resampling


#### Mechanism of Filter Methods:

$$\chi^2\text{-Test}$$

- $\chi^2$-Test: Test for independence between categorical $x_j$ and categorical $y$
- Numeric features or targets can be discretized
```python
from sklearn.preprocessing import KBinsDiscretizer
```
- Hypotheses: 
$$
\begin{align*}
H_0:&p(x_j=m,y=k)=p(x_j=m)p(y=k) \hspace{2 mm} \forall m,k \text{ vs } \\

H_1:&p(x_j=m,y=k)\neq(x_j=m)p(y=k) \\

&\chi_j^2 = \sum_{m=1}^M \sum_{k=1}^K \left(\frac{e_{mk}-\tilde{e}_{mk}}{\tilde{e}_{mk}}\right) \underset{\text{approx.}}{\overset{H_0}{\sim}} \chi^2_{(M-1)(K-1)} \\
\end{align*}
$$

- $e_{mk}$ is observed relative frequency of pair $(m,k)$
- $\tilde{e}_{mk}=\frac{e_m \cdot e_k}{n}$ is expected relative frequency 
- $M$ and $K$ are number of possible values of $x_j$ and $y$
- Large $\chi_j^2$ indicates higher dependence between feature $x_j$ and target $y$ meaning higher relevance of $x_j$ 

```python
from sklearn.feature_selection import chi2
scores, pvals = chi2(X, y)
```

---

$$\text{Pearson Correlation }r(x_j,y)$$
- For numeric features and numeric targets
- checks linear dependance
$$
r(x_j,y)=\frac{\overbrace{\frac{1}{n-1}\sum_{i=1}^n (x_j^{(i)}-\bar{x}_j)(y^{(i)}-\bar{y})}^{\text{Cov}(x_j,y)}}{\sqrt{\underbrace{\frac{1}{n-1}\sum_{i=1}^n (x_j^{(i)}-\bar{x}_j)^2}_{\text{Var}(x_j)}} \sqrt{\underbrace{\frac{1}{n-1}\sum_{i=1}^n (y^{(i)}-\bar{y})^2}_{\text{Var}(y)}}}
$$

- $r(x_j,y) \in [-1,1]$
- $r(x_j,y)$ closer to -1 means negative linear dependency between $x_j$ and $y$
- $r(x_j,y)$ closer to +1 means positive linear dependency between $x_j$ and $y$
- $r(x_j,y)$ closer to 0 means no linear dependency between $x_j$ and $y$
- Use absolute values $|r(x_j,y)|$ for feature ranking since direction of linear dependency is irrelevant


$$\text{Spearman Correlation }r_{SP}(x_j,y)$$
- For features and targets at least on ordinal scale
- like Preason correlation but comouted on ranks
- checks monotonicity of relationship
$$
r_{SP}(x_j,y)=
\frac{
\frac{1}{n-1}\sum_{i=1}^n
\left(R(x_j^{(i)})-\overline{R_x}\right)
\left(R(y^{(i)})-\overline{R_y}\right)
}{
\sqrt{
\frac{1}{n-1}\sum_{i=1}^n
\left(R(x_j^{(i)})-\overline{R_x}\right)^2
}
\sqrt{
\frac{1}{n-1}\sum_{i=1}^n
\left(R(y^{(i)})-\overline{R_y}\right)^2
}}
$$

```python
import pandas as pd

corr = X.corrwith(y) # Pearson
spearman = X.corrwith(y, method="spearman")
```

---
$$\text{Welch's t-test aka two-sample-t-test for samples with unequal variances}$$

- For numeric features and binary targets $y \in \{0,1\}$
- Hypotheses: 
$$
H_0:\mu_{j_0}=\mu_{j_1} \hspace{1 mm} \text{ vs } \hspace{1 mm} H_1:\mu_{j_0}\neq\mu_{j_1} 
$$

$$
t_j = \frac{\bar{x}_{j_0}-\bar{x}_{j_1}}{\sqrt{\frac{S_{x_{j_0}}^2}{n_0} + \frac{S_{x_{j_1}}^2}{n_1}}}
$$

- $\bar{x}_{j_0}$ and $\bar{x}_{j_1}$ are sample mean for $y=0$ and $y=1$
- $S_{x_{j_0}}^2$ are sample variance for $y=0$ and $y=1$
- $n_0$ and $n_1$ are sample sizes for $y=0$ and $y=1$
- Higher value for $t_j$ indicates higher relevance of $x_j$

```python
from scipy.stats import ttest_ind
t_stat, p_val = ttest_ind(X[y==0], X[y==1], equal_var=False)
```

---
$$\text{AUC/ROC}$$

- For binary classification with $y \in \{0,1\}$ and numeric features
- Compute AUC score for every feature (with tresholds) and its ability to separate classes
- Rank features and features with higher AUC score mean higher relevance

```python
from sklearn.metrics import roc_auc_score
roc_auc_score(y, X[:, j])
```

---
$$\text{F-test}$$

- For multiclass classification ($g \geq 2$) and numeric features
- Measures how much the expected values per feature $x_j$ within the classes of the target variable differ from each other
- Hypotheses: 
$$
H_0:\mu_{j_0}=\mu_{j_1}=\ldots=\mu_{j_g}= \hspace{1 mm} \text{ vs } \hspace{1 mm} H_1:\exists k,l: \mu_{j_k}\neq\mu_{j_l}
$$

$$
\begin{align*}
F &= \frac{\text{between-group variability}}{\text{within-group variability}} \\
&= \frac{\sum_{k=1}^g n_k (\bar{x}_{j_k}-\bar{x}_j)^2 / (g-1)}{\sum_{k=1}^g \sum_{i=1}^{n_k}(x_{j_k}^{(i)}-\bar{x}_{j_k})^2/(n-g)}
\end{align*}
$$

- $\bar{x_{j_k}}$ is sample mean of feature $x_j$ where $y=k$
- $\bar{x}_j$ is overall sample mean of feature $x_j$
- High F-score means higher relevance of the feature $x_j$

```python
from sklearn.feature_selection import f_classif
scores, pvals = f_classif(X, y)
```

---
$$\text{Mutual information (MI)}$$
- Each feature $x_j$ is rated according to $\underbrace{I(X,Y)=\mathbb{E}_{p(x,y)}\left[\log\left(\frac{p(X,Y)}{p(X)p(Y)}\right)\right]}_{\text{Information gain}}$

- Estimate: $I(X,Y) = \sum_{x \in X} \sum_{y \in Y} p(x,y)\log\left(\frac{p(X,Y)}{p(X)p(Y)}\right)$ 

- Measures amount of dependence by measuring difference between joint distribution $p(X,Y)$ from their independence $p(X)p(Y)$
- $p(X)$ and $p(Y)$ are marginal probabilities

- MI = 0 if $X$ is independent from $Y$ 
- MI is large when $X$ and $Y$ are dependent or vice versa
- MI is for both numeric and categorical features and targets
- Estimate $p(X,Y)$, $p(X)$ and $p(Y)$ as observed frequencies (for discrete features)
- Estimate $p(X,Y)$, $p(X)$ and $p(Y)$ through binning or kernel density estimation (for continuous features)

```python
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X, y)

from sklearn.metrics import mutual_info_score
```

---

<div style="display:flex; gap:20px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">


##### Advantages
- scales well with increasing number of features $p$
- generally interpretable

</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">

##### Disadvantages

- Filters can be misleading
- Redundant features may still add value when combined with others
- Some features useless alone may be useful together 


</div>
</div>


In [ ]:
# Mutual Information
from collections import Counter
X = np.array([0, 0, 1, 1, 1, 0])
y = np.array([0, 0, 1, 1, 0, 1])
n = len(X)

# Joint distribution p(x,y)
# X | y | Count | Probability |
# - | - | ----- | ----------- |
# 0 | 0 | 2     | 2/6         |
# 0 | 1 | 1     | 1/6         |
# 1 | 0 | 1     | 1/6         |
# 1 | 1 | 2     | 2/6         |

# p(X=0)=3/6
# p(X=1)=3/6
# p(y=0)=3/6
# p(y=1)=3/6

# Compute marginals
px = Counter(X)
py = Counter(y)

# Normalize to probabilities
for k in px:
    px[k] /= n

for k in py:
    py[k] /= n


# Compute joint probabilities
joint = Counter(zip(X,y))

for k in joint:
    joint[k] /= n


# Mutual Information
mi = 0

for (x_val, y_val), pxy in joint.items():

    mi += pxy * np.log(pxy / (px[x_val] * py[y_val]))

print("Mutual Information =", mi)


print("sklearn Mutual Information:",mutual_info_score(X, y))

Mutual Information = 0.056633012265132426
sklearn Mutual Information: 0.0566330122651324


In [ ]:
# KBinsDiscretizer
X = np.array([1,2,3,4,5,6,7,8,9,10])
# With 10 values lets create 5 bins: 

# Bin | Values |
# --- | ------ |
# 0   | 1,2    |
# 1   | 3,4    |
# 2   | 5,6    |
# 3   | 7,8    |
# 4   | 9,10   |

# Replace by Bin IDs
# Sorted values: [1,2,3,4,5,6,7,8,9,10]
#                [0,0,1,1,2,2,3,3,4,4]

n_bins = 5

# Compute quantile boundaries

quantiles = np.percentile(X,np.linspace(0,100,n_bins+1))

print("Quantile edges:",quantiles)


# Assign bins manually
bins = np.digitize(X,quantiles[1:-1], # internal boundaries
                   right=True)

print("Binned values:",bins)


kbd = KBinsDiscretizer(n_bins=5,encode="ordinal",strategy="quantile")

X_binned = kbd.fit_transform(X.reshape(-1,1))

print("\nsklearn Binned values:",X_binned.astype(int))

Quantile edges: [ 1.   2.8  4.6  6.4  8.2 10. ]
Binned values: [0 0 1 1 2 2 3 3 4 4]

sklearn Binned values: [[0]
 [0]
 [1]
 [1]
 [2]
 [2]
 [3]
 [3]
 [4]
 [4]]


c:\Users\iyke\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [ ]:
# Load dataset
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Shape of X:", X.shape)
print("Classes:", np.unique(y))

# Pearson Correlation
print("\n" + "="*60)
print("Pearson Correlation")
print("="*60)

pearson_scores = X.corrwith(y)

pearson_df = pd.DataFrame({"feature": X.columns,"pearson_corr": pearson_scores,"abs_corr": np.abs(pearson_scores)}).sort_values("abs_corr", ascending=False)
print(pearson_df.head(10))


# Spearmann correlation
print("\n" + "="*60)
print("Spearmann correlation")
print("="*60)

spearman_scores = X.corrwith(y, method="spearman")
spearman_df = pd.DataFrame({"feature": X.columns,"spearman_corr": spearman_scores,"abs_corr": np.abs(spearman_scores)}).sort_values("abs_corr", ascending=False)
print(spearman_df.head(10))

# Chi-Square Test
# chi2 requires NON-NEGATIVE features
# Discretize continuous variables first
print("\n" + "="*60)
print("Chi-Square Test")
print("="*60)

discretizer = KBinsDiscretizer(n_bins=5,encode="ordinal",strategy="quantile")

X_binned = discretizer.fit_transform(X)

chi2_scores, chi2_pvals = chi2(X_binned, y)

chi2_df = pd.DataFrame({"feature": X.columns,"chi2_score": chi2_scores,"p_value": chi2_pvals}).sort_values("chi2_score", ascending=False)
print(chi2_df.head(10))


# Welch's t-Tests
print("\n" + "="*60)
print("Welch's t-Tests")
print("="*60)

t_scores = []
p_values = []

for col in X.columns:
    
    group0 = X.loc[y == 0, col]
    group1 = X.loc[y == 1, col]

    t_stat, p_val = ttest_ind(group0,group1,equal_var=False)

    t_scores.append(t_stat)
    p_values.append(p_val)

ttest_df = pd.DataFrame({"feature": X.columns,"t_statistic": t_scores,"abs_t": np.abs(t_scores),"p_value": p_values}).sort_values("abs_t", ascending=False)
print(ttest_df.head(10))


# AUC / ROC
print("\n" + "="*60)
print("AUC / ROC")
print("="*60)

auc_scores = []

for col in X.columns:
    auc = roc_auc_score(y, X[col])
    auc_scores.append(auc)

auc_df = pd.DataFrame({"feature": X.columns,"auc_score": auc_scores,"abs_distance_from_0.5": np.abs(np.array(auc_scores) - 0.5)}).sort_values("abs_distance_from_0.5", ascending=False)
print(auc_df.head(10))


# F-Tests
print("\n" + "="*60)
print("F-Tests")
print("="*60)

f_scores, f_pvals = f_classif(X, y)

f_df = pd.DataFrame({"feature": X.columns,"f_score": f_scores,"p_value": f_pvals}).sort_values("f_score", ascending=False)
print(f_df.head(10))


# Mutual Information
print("\n" + "="*60)
print("Mutual Information")
print("="*60)

mi_scores = mutual_info_classif(X, y, random_state=42)

mi_df = pd.DataFrame({"feature": X.columns,"mutual_information": mi_scores}).sort_values("mutual_information", ascending=False)
print(mi_df.head(10))


# Top Features
print("\n" + "="*60)
print("Top Feature per method")
print("="*60)

print("Pearson:", pearson_df.iloc[0]["feature"])
print("Spearman:", spearman_df.iloc[0]["feature"])
print("Chi2:", chi2_df.iloc[0]["feature"])
print("T-test:", ttest_df.iloc[0]["feature"])
print("AUC:", auc_df.iloc[0]["feature"])
print("F-test:", f_df.iloc[0]["feature"])
print("Mutual Info:", mi_df.iloc[0]["feature"])

Shape of X: (569, 30)
Classes: [0 1]

Pearson Correlation
                                   feature  pearson_corr  abs_corr
worst concave points  worst concave points     -0.793566  0.793566
worst perimeter            worst perimeter     -0.782914  0.782914
mean concave points    mean concave points     -0.776614  0.776614
worst radius                  worst radius     -0.776454  0.776454
mean perimeter              mean perimeter     -0.742636  0.742636
worst area                      worst area     -0.733825  0.733825
mean radius                    mean radius     -0.730029  0.730029
mean area                        mean area     -0.708984  0.708984
mean concavity              mean concavity     -0.696360  0.696360
worst concavity            worst concavity     -0.659610  0.659610

Spearmann correlation
                                   feature  spearman_corr  abs_corr
worst perimeter            worst perimeter      -0.796319  0.796319
worst radius                  worst radius    

c:\Users\iyke\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


                 feature  auc_score  abs_distance_from_0.5
22       worst perimeter   0.024549               0.475451
20          worst radius   0.029557               0.470443
23            worst area   0.030172               0.469828
27  worst concave points   0.033296               0.466704
7    mean concave points   0.035562               0.464438
2         mean perimeter   0.053102               0.446898
3              mean area   0.061684               0.438316
6         mean concavity   0.062173               0.437827
0            mean radius   0.062483               0.437517
13            area error   0.073589               0.426411

F-Tests
                 feature     f_score        p_value
27  worst concave points  964.385393  1.969100e-124
22       worst perimeter  897.944219  5.771397e-119
7    mean concave points  861.676020  7.101150e-116
20          worst radius  860.781707  8.482292e-116
2         mean perimeter  697.235272  8.436251e-101
23            worst area  661.

<a class="anchor" id="wrapper"></a>
## 4.2 Wrapper Methods

**Use models to evaluate subset of features.**

- Use a leaners performance as evaluation metric for a subset of features
- Given $p$ features find subset $S \subset \{1,\ldots,p\}$ optimizing some objective $\psi : \Omega \rightarrow \mathbb{R}:S^* \in \arg \min_{S \in \Omega} \left\{\psi(S)\right\}$
    - $\Omega$ = Search space of all feature subsets $S \subset \{1,\ldots,p\}$
        - usually encoded by bit vectors, i.e. $\Omega=\{0,1\}^p$ (1 = feature selected)
    - Objective $\psi$ can be different functions such cross validated performance of a learner

**Problem:** Discrete combinatorial optimization problem over search space of size = $2^p$, i.e. grows exponentially in $p$.

**Solution:** Avoid searching entire space by employing efficient search strategies.




## Greedy forward search (GFS)
**Start empty and add best feature iteratively and terminate if performance does not improve further or max number of fratures is used.**

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/52.png" width="550"/>
  <img src="pics/53.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/54.png" width="550"/>
  <img src="pics/55.png" width="550"/>
</div>



## Greedy backward search (GBS)
**Start full and remove worst feature iteratively.**

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/56.png" width="550"/>
  <img src="pics/57.png" width="550"/>
</div>


```python
# Forward / Backward Selection
from sklearn.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

sfs = SFS(model, n_features_to_select=10, direction="forward")
X_new = sfs.fit_transform(X, y)

```

```python

# Recursive Feature Elimination (RFE)
from sklearn.feature_selection import RFE

selector = RFE(model, n_features_to_select=10)
X_new = selector.fit_transform(X, y)
```

```python

# More advanced (mlxtend)
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

sfs = SFS(model, k_features=10, forward=True)
sfs.fit(X, y)
```



## Extensions Algorithms (EAs)
**Wrapper feature selection methods can be accelerated and improved by adding/removing multiple features simultaneously, alternating forward and backward search, randomly sampling feature subsets, and focusing the search on promising regions of the feature space.**

<p align="center">
<img src="pics/58.png" width="500"/>
</p>

```python
from deap import base, creator, tools
# custom implementation required
```



<div style="display:flex; gap:20px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">


##### Advantages
- Can be combined with any learner
- Any performance measure can be used
- Optimizes desired criterion directly

</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">

##### Disadvantages

- Does not scale well with growing numbers of features $p$
- Computationlly expensive
- Requires nested resampling


</div>
</div>

In [28]:
# Greed Forward Search (GFS) with Logistic Regression

# Toy data
X, y = make_classification(n_samples=300,n_features=8,n_informative=4,n_redundant=2,random_state=2024)

# Put into dataframe
feature_names = [f"x{i}" for i in range(X.shape[1])]

X = pd.DataFrame(X, columns=feature_names)

#print(X.head())


# Greedy Forward Selection
# Remaining candidate features
remaining_features = list(X.columns)

# Selected subset S
selected_features = []

# Store results
history = []

# Logistic regression model
model = LogisticRegression(max_iter=500)


# Iterative Search

iteration = 1

while len(remaining_features) > 0:

    best_feature = None
    best_risk = np.inf


    # Try adding every remaining feature
    for feature in remaining_features:

        candidate_subset = selected_features + [feature]


        # Estimate risk using CV classification error
        scores = cross_val_score(model,X[candidate_subset],y,cv=5,scoring="accuracy")
        accuracy = scores.mean()

        # Risk = classification error
        risk = 1 - accuracy

        # Keep best feature
        if risk < best_risk:

            best_risk = risk
            best_feature = feature

    # Add best feature permanently
    selected_features.append(best_feature)

    remaining_features.remove(best_feature)

    # Save iteration results
    history.append({"iteration": iteration,"risk": round(best_risk, 4),"selected_subset": selected_features.copy()})
    iteration += 1


# Results Table
results_df = pd.DataFrame(history)

print("\n")
print("*"*70)
print("Greedy Forward Selection results")
print("*"*70)

print(results_df)


# Best Subset
best_idx = results_df["risk"].idxmin()

best_subset = results_df.loc[best_idx, "selected_subset"]

best_risk = results_df.loc[best_idx, "risk"]

print("\n")
print("-"*70)
print("Best subset")
print("-"*70)

print("Features:", best_subset)
print("Risk:", best_risk)



**********************************************************************
Greedy Forward Selection results
**********************************************************************
   iteration    risk                   selected_subset
0          1  0.2367                              [x7]
1          2  0.2067                          [x7, x4]
2          3  0.1733                      [x7, x4, x0]
3          4  0.1733                  [x7, x4, x0, x3]
4          5  0.1800              [x7, x4, x0, x3, x2]
5          6  0.1833          [x7, x4, x0, x3, x2, x1]
6          7  0.1867      [x7, x4, x0, x3, x2, x1, x5]
7          8  0.1867  [x7, x4, x0, x3, x2, x1, x5, x6]


----------------------------------------------------------------------
Best subset
----------------------------------------------------------------------
Features: ['x7', 'x4', 'x0']
Risk: 0.1733


In [42]:
# Greedy Forward Selection (GFS) using sklearn SequentialFeatureSelector

# Toy data
X, y = make_classification(n_samples=300,n_features=8,n_informative=4,n_redundant=2,random_state=1230)

feature_names = [f"x{i}" for i in range(X.shape[1])]

X = pd.DataFrame(X, columns=feature_names)

# Model
model = LogisticRegression(max_iter=500)


# Greedy Forward Selection iteratively
history = []

for k in range(1, X.shape[1]):

    # Sequential Forward Selection
    sfs = SFS(estimator=model,n_features_to_select=k,direction="forward",scoring="accuracy",cv=5)
    sfs.fit(X, y)


    # Selected features
    selected_features = list(X.columns[sfs.get_support()])


    # Compute CV accuracy on selected subset
    scores = cross_val_score(model,X[selected_features],y,cv=5,scoring="accuracy")
    accuracy = scores.mean()

    # Risk = classification error
    risk = 1 - accuracy


    # Store results
    history.append({"iteration": k,"risk": round(risk, 4),"selected_subset": selected_features})


# Results
results_df = pd.DataFrame(history)

print("\n")
print("*"*70)
print("SequentialFeatureSelector Results")
print("*"*70)

print(results_df)


# Best subset
best_idx = results_df["risk"].idxmin()

best_subset = results_df.loc[best_idx, "selected_subset"]

best_risk = results_df.loc[best_idx, "risk"]

print("\n")
print("-"*70)
print("Best subset")
print("-"*70)

print("Features:", best_subset)
print("Risk:", best_risk)



**********************************************************************
SequentialFeatureSelector Results
**********************************************************************
   iteration    risk               selected_subset
0          1  0.2800                          [x4]
1          2  0.1667                      [x4, x7]
2          3  0.1300                  [x4, x6, x7]
3          4  0.1300              [x2, x4, x6, x7]
4          5  0.1300          [x2, x3, x4, x6, x7]
5          6  0.1300      [x2, x3, x4, x5, x6, x7]
6          7  0.1300  [x0, x2, x3, x4, x5, x6, x7]


----------------------------------------------------------------------
Best subset
----------------------------------------------------------------------
Features: ['x4', 'x6', 'x7']
Risk: 0.13


In [32]:
# Greed Backward Search with Logistic Regression

# Toy data
X, y = make_classification(n_samples=300,n_features=8,n_informative=4,n_redundant=2,random_state=2110)

# Put into dataframe
feature_names = [f"x{i}" for i in range(X.shape[1])]

X = pd.DataFrame(X, columns=feature_names)
#print(X.head())


# Greed Backward Selection

# Start with FULL feature set
selected_features = list(X.columns)

# Store results
history = []

# Logistic regression model
model = LogisticRegression(max_iter=500)


# Iterative Search
iteration = 1

while len(selected_features) > 1:

    best_subset = None
    best_risk = np.inf
    removed_feature = None


    # Try removing every feature
    for feature in selected_features:

        candidate_subset = [f for f in selected_features if f != feature]


        # Estimate risk using CV classification error
        scores = cross_val_score(model,X[candidate_subset],y,cv=5,scoring="accuracy")
        accuracy = scores.mean()

        # Risk = classification error
        risk = 1 - accuracy

        # Keep best subset
        if risk < best_risk:

            best_risk = risk
            best_subset = candidate_subset
            removed_feature = feature


    # Permanently remove worst feature
    selected_features = best_subset

    # Save iteration results
    history.append({"iteration": iteration,"removed_feature": removed_feature,"risk": round(best_risk, 4),"selected_subset": selected_features.copy()})

    iteration += 1

# Results
results_df = pd.DataFrame(history)

print("\n")
print("*"*70)
print("Greedy Backward Selection Results")
print("*"*70)

print(results_df)


# Best subset
best_idx = results_df["risk"].idxmin()

best_subset = results_df.loc[best_idx, "selected_subset"]

best_risk = results_df.loc[best_idx, "risk"]

print("\n")
print(">"*70)
print("Best subset")
print(">"*70)

print("Features:", best_subset)
print("Risk:", best_risk)



**********************************************************************
Greedy Backward Selection Results
**********************************************************************
   iteration removed_feature    risk               selected_subset
0          1              x0  0.2367  [x1, x2, x3, x4, x5, x6, x7]
1          2              x2  0.2367      [x1, x3, x4, x5, x6, x7]
2          3              x1  0.2433          [x3, x4, x5, x6, x7]
3          4              x6  0.2400              [x3, x4, x5, x7]
4          5              x5  0.2533                  [x3, x4, x7]
5          6              x7  0.2833                      [x3, x4]
6          7              x4  0.3933                          [x3]


>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
Best subset
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
Features: ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7']
Risk: 0.2367


In [ ]:
# GREEDY BACKWARD SELECTION (GBS)using sklearn SequentialFeatureSelector

# Toy data
X, y = make_classification(n_samples=300,n_features=8,n_informative=4,n_redundant=2,random_state=1429)

# Put into dataframe
feature_names = [f"x{i}" for i in range(X.shape[1])]

X = pd.DataFrame(X, columns=feature_names)


# Model
model = LogisticRegression(max_iter=500)


# Greedy Backward Selection iteratively
history = []

n_features = X.shape[1]

# Start with all features and reduce iteratively
for k in range(n_features - 1, 0, -1): # loop backwards


    # Sequential Backward Selection
    sfs = SFS(estimator=model,n_features_to_select=k,direction="backward",scoring="accuracy",cv=5)
    sfs.fit(X, y)

  
    # Selected subset
    selected_features = list(X.columns[sfs.get_support()])


    # Determine removed feature
    removed_feature = list(set(X.columns) - set(selected_features))


    # Compute CV accuracy
    scores = cross_val_score(model,X[selected_features],y,cv=5,scoring="accuracy")
    accuracy = scores.mean()

    # Risk = classification error
    risk = 1 - accuracy


    # Store results
    history.append({"iteration": n_features - k,"removed_feature(s)": removed_feature,"risk": round(risk, 4),"selected_subset": selected_features})


# Results
results_df = pd.DataFrame(history)

print("\n")
print("*"*70)
print("Sequential Backward Selection Results")
print("*"*70)

print(results_df)


# Best subset
best_idx = results_df["risk"].idxmin()

best_subset = results_df.loc[best_idx, "selected_subset"]

best_risk = results_df.loc[best_idx, "risk"]

print("\n")
print(">"*70)
print("Best subset")
print(">"*70)

print("Features:", best_subset)
print("Risk:", best_risk)



**********************************************************************
Sequential Backward Selection Results
**********************************************************************
   iteration            removed_feature(s)    risk  \
0          1                          [x1]  0.1233   
1          2                      [x5, x1]  0.1233   
2          3                  [x2, x5, x1]  0.1200   
3          4              [x0, x2, x5, x1]  0.1200   
4          5          [x5, x1, x7, x2, x0]  0.1200   
5          6      [x5, x1, x7, x2, x0, x6]  0.1233   
6          7  [x1, x5, x7, x4, x2, x0, x6]  0.1300   

                selected_subset  
0  [x0, x2, x3, x4, x5, x6, x7]  
1      [x0, x2, x3, x4, x6, x7]  
2          [x0, x3, x4, x6, x7]  
3              [x3, x4, x6, x7]  
4                  [x3, x4, x6]  
5                      [x3, x4]  
6                          [x3]  


>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
Best subset
>>>>>>>>>>>>>>>>>>>>>>>>>>>>

In [38]:
# EAs

# Demonstrates:
# - Bit-vector encoding of feature subsets
# - Population initialization
# - Fitness evaluation using CV accuracy
# - Parent selection
# - Crossover
# - Mutation
# - Evolution over generations


# Toy data
X, y = make_classification(n_samples=300,n_features=10,n_informative=4,n_redundant=2,random_state=1200)

feature_names = [f"x{i}" for i in range(X.shape[1])]

X = pd.DataFrame(X, columns=feature_names)


# Hyperparameters
POPULATION_SIZE = 8 # mu
OFFSPRING_SIZE = 6 # lambda
GENERATIONS = 10

MUTATION_RATE = 0.1


# Model
model = LogisticRegression(max_iter=500)


# Initialize random Population
# Bit vector:
# 1 = feature selected
# 0 = feature not selected
#
# Example:
# [1,0,1,0,1]

n_features = X.shape[1]

population = []

for _ in range(POPULATION_SIZE):

    chromosome = np.random.randint(0,2,size=n_features)

    # Avoid empty feature subset
    if chromosome.sum() == 0:
        chromosome[np.random.randint(0, n_features)] = 1

    population.append(chromosome)


# Fitness Function
def evaluate_fitness(chromosome):

    # Selected feature indices
    selected_idx = np.where(chromosome == 1)[0]

    # Prevent empty subset
    if len(selected_idx) == 0:
        return 0

    selected_features = X.columns[selected_idx]

    scores = cross_val_score(model,X[selected_features],y,cv=5,scoring="accuracy")

    return scores.mean()


# Crossover
def crossover(parent1, parent2):

    point = np.random.randint(1, n_features - 1)

    child1 = np.concatenate([parent1[:point],parent2[point:]])

    child2 = np.concatenate([parent2[:point],parent1[point:]])

    return child1, child2


# Mutation
def mutate(chromosome):

    for i in range(len(chromosome)):

        if np.random.rand() < MUTATION_RATE:

            chromosome[i] = 1 - chromosome[i]

    # Avoid empty subset
    if chromosome.sum() == 0:
        chromosome[np.random.randint(0, n_features)] = 1

    return chromosome


# Evolution Loop
history = []

for generation in range(GENERATIONS):


    # Evaluate current population
    fitness_scores = np.array([evaluate_fitness(chrom)for chrom in population])


    # Select mu fittest parents
    parent_indices = np.argsort(fitness_scores)[::-1][:POPULATION_SIZE]

    parents = [population[i] for i in parent_indices]


    # Generate lambda offspring
    offspring = []

    while len(offspring) < OFFSPRING_SIZE:

        p1, p2 = random.sample(parents, 2)

        child1, child2 = crossover(p1, p2)

        child1 = mutate(child1)
        child2 = mutate(child2)

        offspring.append(child1)

        if len(offspring) < OFFSPRING_SIZE:
            offspring.append(child2)


    # Combine parents + offspring
    combined_population = parents + offspring

    combined_fitness = np.array([evaluate_fitness(chrom)for chrom in combined_population])


    # Select next generation (mu best)
    next_indices = np.argsort(combined_fitness)[::-1][:POPULATION_SIZE]

    population = [combined_population[i]for i in next_indices]


    # Save best solution
    best_idx = np.argmax(combined_fitness)

    best_chromosome = combined_population[best_idx]

    best_features = list(X.columns[np.where(best_chromosome == 1)[0]])

    best_score = combined_fitness[best_idx]

    history.append({"generation": generation + 1,"best_accuracy": round(best_score, 4),"selected_features": best_features})


# Result
results_df = pd.DataFrame(history)

print("\n")
print("#"*80)
print("GENETIC ALGORITHM FEATURE SELECTION")
print("#"*80)

print(results_df)


# Best solution
best_row = results_df.iloc[results_df["best_accuracy"].idxmax()]

print("\n")
print("#"*80)
print("Best Feature Subset")
print("#"*80)

print("Accuracy:", best_row["best_accuracy"])
print("Features:", best_row["selected_features"])



################################################################################
GENETIC ALGORITHM FEATURE SELECTION
################################################################################
   generation  best_accuracy         selected_features
0           1           0.81      [x0, x1, x2, x4, x8]
1           2           0.81      [x0, x1, x2, x4, x8]
2           3           0.81      [x0, x1, x2, x4, x8]
3           4           0.81      [x0, x1, x2, x4, x8]
4           5           0.82  [x0, x1, x2, x4, x5, x8]
5           6           0.82  [x0, x1, x2, x4, x5, x8]
6           7           0.82  [x0, x1, x2, x4, x5, x8]
7           8           0.82  [x0, x1, x2, x4, x5, x8]
8           9           0.82  [x0, x1, x2, x4, x5, x8]
9          10           0.82  [x0, x1, x2, x4, x5, x8]


################################################################################
Best Feature Subset
################################################################################
Accuracy: 0

<a class="anchor" id="embedded"></a>
## 4.3 Embedded Methods

**Integrate Feature Selection directly into specific model.**

- Example: Lasso with L1 penalty performs automatic feature selction
$$
\begin{align*}
\mathcal{R}_\text{reg}(\theta)&=\mathcal{R}_\text{emp}(\theta) + \lambda \lVert \theta \rVert_1 \\ 
&= \sum_{i=1}^n (y^{(i)}-\theta^\top x^{(i)})^2 + \lambda \sum_{i=1}^p |\theta_j|
\end{align*}
$$

```python
# Lasso (L1 regularization)
from sklearn.linear_model import Lasso

model = Lasso(alpha=0.1)
model.fit(X, y)

selected_features = model.coef_ != 0

# Logistic Regression with L1
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(penalty="l1", solver="liblinear")
model.fit(X, y)


# Tree-based models (feature importance)
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X, y)

importances = model.feature_importances_

```


<a class="anchor" id="insights"></a>
# 5. Key Insights

&#128202; Regularization & High Dimensions
- More features $\rightarrow$ need stronger regularization
- Helps control model complexity

&#127757; Real-world Data (e.g., gene data)
- Good performance requires feature selection or regularization
- Small, well-chosen feature sets can perform best

&#128680; Integrated Selection is Not Always Optimal
- Models with built-in selection don’t always outperform others
- Using only relevant features often improves results significantly

<a class="anchor" id="pipeline"></a>
# 6. How Filter Methods Work (Pipeline)

1. Compute score for each feature
2. Rank features
3. Select top $\tilde{p}$ features
4. Train model

```python
from sklearn.feature_selection import SelectKBest
selector = SelectKBest(score_func=f_classif, k=10) # choose top features based on F-test scores
X_new = selector.fit_transform(X, y)

```



```python

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("fs", SelectKBest(f_classif, k=10)),
    ("model", LogisticRegression())
])
```

Choosing number of features:

- Based on domain knowledge
- Visual inspection
- Hyperparameter tuning

In [49]:
# 1. Compute feature scores
# 2. Rank features
# 3. Select top k features
# 4. Train model
# 5. Hyperparameter tuning for k

# Toy data
X, y = make_classification(n_samples=500,n_features=20,n_informative=5,n_redundant=5,random_state=1509)

feature_names = [f"x{i}" for i in range(X.shape[1])]

X = pd.DataFrame(X, columns=feature_names)
#print(X.head())


# Compute feature scores (F-Test)
selector = SelectKBest(score_func=f_classif,
                       k="all" # compute scores for all features
                       )

selector.fit(X, y)

# F-scores
scores = selector.scores_

# p-values
pvals = selector.pvalues_


# Rank features
ranking_df = pd.DataFrame({"feature": X.columns,"f_score": scores,"p_value": pvals})
ranking_df = ranking_df.sort_values(by="f_score",ascending=False)

print("\n")
print("="*70)
print("Feature ranking")
print("="*70)
print(ranking_df)


# Select top k features
k = 5

selector = SelectKBest(score_func=f_classif,k=k)

X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("\n")
print("="*70)
print(f"Top {k} Features")
print("="*70)

print(list(selected_features))

print("\nShape before selection:", X.shape)
print("Shape after selection :", X_selected.shape)


# Train model on selected features
model = LogisticRegression(max_iter=500)

scores = cross_val_score(model,X[selected_features],y,cv=5,scoring="accuracy")

print("\n")
print("="*70)
print("Model Performance")
print("="*70)

print("CV Accuracy scores:", scores)
print("Mean accuracy:", scores.mean())


# Full Pipeline
pipeline = Pipeline([("fs", SelectKBest(f_classif)),("model", LogisticRegression(max_iter=500))])


# Hyperparameter Tuning for k
param_grid = {"fs__k": [2, 4, 6, 8, 10, 15]}

grid = GridSearchCV(pipeline,param_grid=param_grid,cv=5,scoring="accuracy")
grid.fit(X, y)


# Best pipeline
print("\n")
print("="*70)
print("Hyperparameter Tuning")
print("="*70)

print("Best k:", grid.best_params_["fs__k"])
print("Best CV accuracy:", round(grid.best_score_, 4))


# Best selected features
best_selector = grid.best_estimator_.named_steps["fs"]
best_features = X.columns[best_selector.get_support()]

print("\n")
print("="*70)
print("Best feature subset")
print("="*70)

print(list(best_features))



Feature ranking
   feature    f_score       p_value
17     x17  56.838654  2.255074e-13
11     x11  44.585315  6.517744e-11
3       x3  44.214480  7.755738e-11
8       x8  37.411618  1.936037e-09
0       x0  34.694122  7.106315e-09
9       x9  22.507131  2.738835e-06
2       x2  13.965183  2.078040e-04
16     x16  13.291928  2.945869e-04
1       x1   4.460694  3.518073e-02
7       x7   3.421357  6.495034e-02
18     x18   2.273118  1.322690e-01
6       x6   0.460424  4.977412e-01
12     x12   0.285973  5.930516e-01
5       x5   0.174959  6.759216e-01
13     x13   0.133402  7.150854e-01
14     x14   0.102085  7.494762e-01
15     x15   0.079307  7.783557e-01
10     x10   0.026094  8.717356e-01
19     x19   0.005414  9.413762e-01
4       x4   0.000598  9.804927e-01


Top 5 Features
['x0', 'x3', 'x8', 'x11', 'x17']

Shape before selection: (500, 20)
Shape after selection : (500, 5)


Model Performance
CV Accuracy scores: [0.74 0.73 0.89 0.72 0.73]
Mean accuracy: 0.762


Hyperparameter Tun

<a class="anchor" id="overall"></a>
# 7. Overall Takeaways

- Too many features can hurt performance
- Feature selection is crucial in high-dimensional data
- No single best method:
    - Filters $\rightarrow$ fast but simple
    - Wrappers $\rightarrow$ accurate but expensive
    - Embedded $\rightarrow$ efficient compromise
- Feature interactions are important $\rightarrow$ univariate methods can miss them
- Feature selection affects both prediction and statistical inference